# Control flow and functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-03-control-flow-and-functions.ipynb)

Pipelines are made of small functions. Each one takes data in, checks it and returns data out. This chapter covers the statements that decide what runs and how often (`if`, `match`, `for`, `while`), how to write functions that are hard to call wrongly, and the scope rules that decide which variable a name refers to. Closures, lambdas and decorators, which are functions that build or wrap other functions, come last.

Every rule is stated before the code that shows it, and each code cell's `assert` lines are the expected results. If an `assert` fails, your model of that rule is wrong.

## Conditionals: `if`, `elif`, `else`

Python checks conditions top to bottom and runs the first branch whose condition is true. Later branches are skipped, so order the tests from the most specific to the most general.

A **conditional expression** `a if condition else b` chooses one of two values and produces a single result. Use it for a value. Use an `if` statement when a branch has several statements.

`match` (Python 3.10 and later) compares a value against **patterns**. A mapping pattern such as `{"type": "order"}` matches any dictionary that has that key, even if it has other keys too. The `case _:` branch is the default.

In [ ]:
def grade(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    else:
        return "C"

assert grade(95) == "A" and grade(85) == "B" and grade(10) == "C"

# A conditional expression chooses one value.
label = "large" if 150 > 100 else "small"
assert label == "large"

def handle(event):
    match event:
        case {"type": "order", "id": order_id}:
            return f"order {order_id}"
        case {"type": "refund"}:           # matches even when extra keys are present
            return "refund"
        case _:
            return "ignored"

assert handle({"type": "order", "id": 7}) == "order 7"
assert handle({"type": "refund", "amount": 5}) == "refund"
assert handle({"type": "ping"}) == "ignored"

## `for`, `range` and `while`

`for` walks any iterable. `range(start, stop, step)` produces integers, and **`stop` is never included**. `range(1, 4)` is `1, 2, 3`. A negative `step` counts down.

`while` repeats while its condition is true. It checks the condition before each pass, so it may run zero times. Every `while` loop must change something that makes the condition false, or it never ends.

In [ ]:
assert list(range(1, 11, 2)) == [1, 3, 5, 7, 9]    # stop (11) is excluded
assert list(range(5, 0, -2)) == [5, 3, 1]          # a negative step counts down
assert list(range(0)) == []                        # an empty range runs zero times

total = 0
for n in range(1, 11, 2):
    total += n
assert total == 25

# while checks the condition first, so this body runs zero times.
runs = 0
while runs > 0:
    runs -= 1
assert runs == 0

# A while loop that makes progress toward its exit condition.
attempts = 0
while attempts < 3:
    attempts += 1
assert attempts == 3

### Where you'll use it

- **Row-by-row validation:** a loop checks each record and routes it to the load or to the quarantine.
- **Retry loops:** a `while` with a counter waits for a file that has not landed yet, and gives up after a fixed number of checks.

## `enumerate`, `zip` and their loop-safe forms

`enumerate(items, start=1)` gives `(number, item)` pairs, so you never keep a counter by hand. `zip(a, b)` walks two sequences together and **stops at the shorter one**, so a short input silently drops rows. Add `strict=True` (Python 3.10 and later) to raise an error when the lengths differ. The collections chapter covers this in more depth.

In [ ]:
names = ["ana", "ben", "cy"]
for index, name in enumerate(names, start=1):
    if index == 2:
        assert name == "ben"

assert list(zip(names, [30, 25, 41]))[2] == ("cy", 41)
assert list(zip(names, [30, 25])) == [("ana", 30), ("ben", 25)]     # cy is dropped

try:
    list(zip(names, [30, 25], strict=True))
except ValueError:
    pass
else:
    raise AssertionError("strict=True should refuse unequal lengths")

## `break`, `continue` and `for ... else`

`break` leaves the loop at once. `continue` skips to the next item. The `else` block of a loop runs only when the loop finished **without** `break`. It is the clean way to write "search, and do something if nothing was found", without a flag variable.

### Where you'll use it

- **Skipping bad rows:** `continue` moves past a record that fails validation without stopping the file.
- **Searching for a marker:** `break` stops reading once a footer line or end-of-batch marker is found.

In [ ]:
def find_first_negative(values):
    for index, value in enumerate(values):
        if value < 0:
            found = index
            break
    else:
        found = None          # the loop ended without break: nothing is negative
    return found

assert find_first_negative([3, -1, -5]) == 1
assert find_first_negative([1, 2]) is None

kept = []
for value in [1, None, 3]:
    if value is None:
        continue
    kept.append(value)
assert kept == [1, 3]

## Defining functions

A function does one job. It takes inputs as parameters, and it returns a value with `return`. A function without `return` returns `None`. Write a **docstring** as the first statement of the body, so `help()` and readers see the contract.

A function can return several values. Python packs them into a tuple, and the caller unpacks them.

In [ ]:
def min_max(values):
    """Return the smallest and largest value as a (low, high) tuple."""
    return min(values), max(values)

low, high = min_max([3, 9, 1])
assert (low, high) == (1, 9)
assert min_max.__doc__.startswith("Return the smallest")

def no_return():
    pass

assert no_return() is None

## Parameter kinds: positional, keyword and keyword-only

Parameters can be passed **by position** (`f(1, 2)`) or **by keyword** (`f(b=2)`). Two markers control this:

- A `/` in the signature makes the parameters before it **positional-only**. They cannot be passed by name.
- A `*` in the signature makes the parameters after it **keyword-only**. A caller must name them, which stops a boolean in the wrong position from changing the behaviour.

In [ ]:
def load_window(start, end, *, inclusive_end=False):
    """Return the window as a (start, stop) tuple. stop is exclusive unless inclusive_end is set."""
    stop = end + 1 if inclusive_end else end
    return start, stop

assert load_window(1, 5) == (1, 5)
assert load_window(1, 5, inclusive_end=True) == (1, 6)

try:
    load_window(1, 5, True)       # the flag must be passed by name
except TypeError:
    pass
else:
    raise AssertionError("expected TypeError")

def clamp(value, /, low, high):
    return max(low, min(value, high))

assert clamp(15, 0, 10) == 10
assert clamp(-3, low=0, high=10) == 0      # low and high may be passed by name

try:
    clamp(value=15, low=0, high=10)        # value is positional-only
except TypeError:
    pass
else:
    raise AssertionError("expected TypeError")

### Where you'll use it

- **Transformation steps:** each pipeline step is a function from input records to output records, which makes each step testable and reusable.
- **Keyword-only options:** `write(path, *, overwrite=False)` prevents a boolean in the wrong position from overwriting a file.

## `*args`, `**kwargs` and unpacking in calls

`*args` collects extra positional arguments into a **tuple**. `**kwargs` collects extra keyword arguments into a **dict**. Use them to forward arguments through a wrapper, not to hide what a function accepts.

In a call, `*` and `**` do the reverse: they spread a sequence or a dict into separate arguments.

In [ ]:
def connect(host, *args, timeout=5, **options):
    return host, args, timeout, options

host, args, timeout, options = connect("db1", 1, 2, timeout=10, retries=3)
assert host == "db1"
assert args == (1, 2)
assert timeout == 10
assert options == {"retries": 3}

settings = {"timeout": 30}
assert connect("db2", **settings)[2] == 30       # ** spreads a dict into keyword arguments

coordinates = [1, 2]
assert connect(*["db3", *coordinates])[1] == (1, 2)   # * spreads a list into positional arguments

### Where you'll use it

- **Wrappers and decorators:** a logging or retry wrapper forwards every argument to the function it wraps.
- **Connector options:** database drivers and HTTP clients accept extra keyword options that you pass through unchanged.

## The mutable default argument trap

A default value is created **once**, when the `def` statement runs, not on each call. A mutable default such as `[]` is shared by every call that uses it. Use `None` as the default and create the object inside the function.

### Where you'll use it

- **Shared-state bugs in batch jobs:** a default list that collects rows across runs in one long-lived worker. It appears only after the second run, so tests often miss it.

In [ ]:
def add_bad(item, bucket=[]):          # the list is created once, at definition time
    bucket.append(item)
    return bucket

def add_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

add_bad("a")
assert add_bad("b") == ["a", "b"]      # the shared default still holds "a"

assert add_good("a") == ["a"]
assert add_good("b") == ["b"]          # each call starts fresh

## Scope: which variable does a name mean?

Python looks up a name in this order: the current function (**L**ocal), any enclosing functions (**E**nclosing), the module (**G**lobal), then the built-ins (**B**uilt-in). This is the LEGB rule.

Assigning a name **anywhere** inside a function makes it local for the whole function, even if the assignment comes after a line that reads it. Reading a local before its assignment raises `UnboundLocalError`. To assign to an outer function's variable, declare it `nonlocal`. Avoid `global` in pipeline code, because it makes a function depend on hidden state.

In [ ]:
greeting = "global"

def read_only():
    return greeting               # reads the module-level name

def shadow():
    greeting = "local"            # creates a new local name; the global is untouched
    return greeting

assert read_only() == "global"
assert shadow() == "local"
assert greeting == "global"

def broken():
    retry_budget = retry_budget + 1    # the assignment makes retry_budget local for the whole function

try:
    broken()
except UnboundLocalError:
    pass
else:
    raise AssertionError("expected UnboundLocalError")

## Closures: functions that remember their enclosing scope

A **closure** is an inner function that uses variables from the function that created it. The inner function keeps those variables alive after the outer function returns. Use closures to build a configured function, or a small piece of state, without a class.

### Where you'll use it

- **Configured callables:** a function that builds a validator for one column, where the column name is remembered by the closure.
- **Stateful batching:** a `next_batch` function that remembers its position in a stream.

In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor           # factor is remembered after make_multiplier returns
    return multiply

double = make_multiplier(2)
assert double(21) == 42

def make_counter():
    count = 0
    def increment():
        nonlocal count                  # rebind the enclosing variable, not a new local
        count += 1
        return count
    return increment

counter = make_counter()
counter()
assert counter() == 2

### Late binding: the loop-closure trap

A closure looks up its variables **when it is called**, not when it is created. Functions created in a loop all share the same loop variable, which has its final value by the time they run. A default argument captures the current value instead.

In [ ]:
late = [lambda: i for i in range(3)]
assert [f() for f in late] == [2, 2, 2]          # every lambda sees the final i

fixed = [lambda i=i: i for i in range(3)]        # the default is evaluated now, once per lambda
assert [f() for f in fixed] == [0, 1, 2]

## Lambdas, `map`, `filter`, `reduce` and `partial`

A `lambda` is an anonymous function with one expression. Use it for a short `key=` function. Write a `def` for anything longer.

- `map(f, items)` and `filter(pred, items)` return **lazy iterators**, so wrap them in `list()` to see the values.
- `functools.reduce(f, items, start)` folds a sequence into one value.
- `functools.partial(f, **fixed)` returns a new function with some arguments already set.

A comprehension is often clearer than `map` or `filter`. Use whichever reads better.

In [ ]:
from functools import partial, reduce

rows = [{"amount": 3}, {"amount": 10}, {"amount": 7}]

assert list(map(lambda r: r["amount"] * 2, rows)) == [6, 20, 14]
assert list(filter(lambda r: r["amount"] > 5, rows)) == [{"amount": 10}, {"amount": 7}]
assert reduce(lambda total, r: total + r["amount"], rows, 0) == 20

def scale(value, factor):
    return value * factor

double = partial(scale, factor=2)
assert double(5) == 10

### Where you'll use it

- **Sorting and grouping keys:** `sorted(rows, key=lambda r: r["updated_at"])`, and the same key pattern in `itertools.groupby`.
- **Column transforms:** passing a small function to a generic helper that applies a rule to every value in a column.

## Decorators: functions that wrap functions

A **decorator** is a function that takes a function and returns a new one with extra behaviour. The `@name` line above a `def` is shorthand for `f = name(f)`. The wrapper receives every argument with `*args, **kwargs`, and calls the original.

Use `functools.wraps(fn)` inside the decorator. Without it, the wrapped function loses its name and docstring, which makes logs and `help()` misleading.

A decorator that takes arguments is a function that returns a decorator, so it has one more level of nesting.

### Where you'll use it

- **Retries and timing:** `@retry(times=3)` on a function that calls a flaky API. Chapter 9 builds one.
- **Logging and metrics:** record the call, its duration and its errors without touching the function body.

In [ ]:
import functools

def log_calls(fn):
    calls = []
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        calls.append((args, kwargs))          # extra behaviour: remember the call
        return fn(*args, **kwargs)            # then run the original function
    wrapper.calls = calls
    return wrapper

@log_calls
def add(a, b):
    """Add two numbers."""
    return a + b

assert add(2, 3) == 5
assert add.calls == [((2, 3), {})]
assert add.__name__ == "add" and add.__doc__ == "Add two numbers."   # wraps kept them

def repeat_on_error(times):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_error = None
            for _ in range(times):
                try:
                    return fn(*args, **kwargs)
                except ValueError as error:
                    last_error = error
            raise last_error
        return wrapper
    return decorator

failures = {"left": 2}

@repeat_on_error(times=3)
def fetch_page():
    if failures["left"] > 0:
        failures["left"] -= 1
        raise ValueError("temporary")
    return "page"

assert fetch_page() == "page"
assert failures["left"] == 0

def shout(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return wrapper

def bracket(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        return f"[{fn(*args, **kwargs)}]"
    return wrapper

@bracket          # applied second: wraps the result of shout
@shout            # applied first: uppercases the result of greet
def greet():
    return "hi"

assert greet() == "[HI]"

## Practice questions

1. Write `first_duplicate(values)` that returns the first value seen twice, or `None`.
2. Write `make_batcher(items, size)` that returns a function. Each call returns the next `size` items of `items`, or `[]` once they are used up.
3. Fix this function so calls do not share state:

   ```python
   def record(event, seen=[]):
       seen.append(event)
       return seen
   ```

4. Write `apply_all(value, *functions)` that applies each function in order, passing the result of one to the next. For example, `apply_all(" Ana ", str.strip, str.lower)` returns `"ana"`.

## Answers

### Question 1

In [ ]:
def first_duplicate(values):
    seen = set()
    for value in values:
        if value in seen:
            return value
        seen.add(value)
    return None

assert first_duplicate([2, 1, 3, 1, 2]) == 1
assert first_duplicate([1, 2]) is None

### Question 2

The closure keeps `position` between calls. `nonlocal` lets it move that position forward.

In [ ]:
def make_batcher(items, size):
    position = 0
    def next_batch():
        nonlocal position
        batch = items[position:position + size]
        position += size
        return batch
    return next_batch

next_batch = make_batcher(list(range(5)), 2)
assert next_batch() == [0, 1]
assert next_batch() == [2, 3]
assert next_batch() == [4]
assert next_batch() == []

### Question 3

In [ ]:
def record(event, seen=None):
    if seen is None:
        seen = []
    seen.append(event)
    return seen

assert record("a") == ["a"]
assert record("b") == ["b"]

### Question 4

In [ ]:
def apply_all(value, *functions):
    for function in functions:
        value = function(value)
    return value

assert apply_all(" Ana ", str.strip, str.lower) == "ana"

## Interview notes

- **`for ... else` is the search idiom.** Its `else` runs only when `break` never happened, so the "not found" branch is explicit.
- **Parameter kinds document the contract.** Keyword-only arguments after `*` make call sites self-explanatory in pipelines with many options.
- **Mutable defaults are shared across calls.** Use `None` and create the object inside the function.
- **Closures capture variables, not values.** A closure built in a loop needs a default argument to freeze each value.
- **LEGB and `UnboundLocalError`.** Any assignment makes a name local to the whole function. Avoid `global`, and use `nonlocal` for the rare case that needs it.
- **`functools.wraps` in every decorator.** Without it, the wrapper hides the original name and docstring.